# Notebook 2: Spark Structured Streaming from Kafka

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Streaming_Pipeline') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.driver.memory', '1g') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.cores.max', '1') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57874fe3-c6d3-4f3a-8361-d240b382d755;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-

Spark version: 3.5.0


## 1. Define Schema

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

# Schema mirrors the JSON produced by kafka_producer.py:
# { "user_id": int, "item_id": int, "rating": float, "timestamp": ISO-8601 string }
schema = StructType([
    StructField('user_id',   IntegerType(), True),
    StructField('item_id',   IntegerType(), True),
    StructField('rating',    FloatType(),   True),
    StructField('timestamp', StringType(),  True),
])

print('Schema defined.')
print(schema.simpleString())

Schema defined.
struct<user_id:int,item_id:int,rating:float,timestamp:string>


## 2. Consume from Kafka

In [3]:
from pyspark.sql.functions import from_json, col, to_timestamp
from pyspark.sql.types import TimestampType

# Read raw bytes from Kafka — value column is binary JSON
raw_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:9092') \
    .option('subscribe', 'user_events') \
    .option('startingOffsets', 'earliest') \
    .load()

# Parse JSON value and cast timestamp string to TimestampType
parsed_df = raw_df \
    .select(from_json(col('value').cast('string'), schema).alias('data')) \
    .select(
        col('data.user_id').alias('user_id'),
        col('data.item_id').alias('item_id'),
        col('data.rating').alias('rating'),
        to_timestamp(col('data.timestamp'), "yyyy-MM-dd'T'HH:mm:ssXXX").alias('event_time')
    ) \
    .filter(
        # Handle malformed records: drop rows where any field failed to parse
        col('user_id').isNotNull() &
        col('item_id').isNotNull() &
        col('rating').isNotNull() &
        col('event_time').isNotNull()
    )

print('Streaming source connected.')
print('Malformed records (null fields after parse) will be silently dropped.')
print('Schema of parsed stream:')
parsed_df.printSchema()

Streaming source connected.
Malformed records (null fields after parse) will be silently dropped.
Schema of parsed stream:
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- event_time: timestamp (nullable = true)



## 3. Apply Watermark

In [4]:
# Watermark tells Spark how long to wait for late-arriving data.
# Events arriving more than 10 seconds after their timestamp are dropped.
# Without this, Spark would keep state forever and eventually run out of memory.
watermarked_df = parsed_df.withWatermark('event_time', '10 seconds')

print('Watermark applied: 10 seconds')
print('  Late data policy: events > 10s past their timestamp are dropped.')

Watermark applied: 10 seconds
  Late data policy: events > 10s past their timestamp are dropped.


## 4. Window Analytics (30s window / 10s slide)

In [5]:
from pyspark.sql.functions import window, avg, count, round as spark_round

# Group by sliding window + item_id
# Window size : 30 seconds  — how much history each aggregate covers
# Slide interval: 10 seconds — how often a new result is emitted
windowed_df = (
    watermarked_df
    .groupBy(
        window(col('event_time'), '30 seconds', '10 seconds'),
        col('item_id')
    )
    .agg(
        spark_round(avg('rating'), 4).alias('avg_rating'),
        count('*').alias('interaction_count')
    )
)

print('Window aggregation defined: 30s window, 10s slide.')
print('  avg_rating        : mean rating for this item in the window')
print('  interaction_count : number of events for this item in the window')

Window aggregation defined: 30s window, 10s slide.
  avg_rating        : mean rating for this item in the window
  interaction_count : number of events for this item in the window


## 5. Custom Metric — Trending Score

In [6]:
# Trending score = interaction_count × avg_rating
# Rationale: a product must be BOTH frequently interacted with AND highly rated
# to surface as trending. High volume alone (spam) or high rating alone (one review)
# will not produce a high score — both signals must be strong simultaneously.
trending_df = windowed_df.withColumn(
    'trending_score',
    spark_round(col('interaction_count') * col('avg_rating'), 4)
)

# Clean, flat output schema for downstream consumers
output_df = trending_df.select(
    col('window.start').alias('window_start'),
    col('window.end').alias('window_end'),
    col('item_id'),
    col('avg_rating'),
    col('interaction_count'),
    col('trending_score')
)

print('Trending score = interaction_count x avg_rating')
print('Custom metric defined.')
print('Output schema:')
output_df.printSchema()

Trending score = interaction_count x avg_rating
Custom metric defined.
Output schema:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- avg_rating: double (nullable = true)
 |-- interaction_count: long (nullable = false)
 |-- trending_score: double (nullable = true)



## 6. Write Stream to Console (for testing)

In [7]:
console_query = (
    output_df
    .writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('numRows', 20)
    .trigger(processingTime='10 seconds')
    .queryName('console_output10')
    .start()
)

print('Streaming query started — waiting 60 seconds for events...')
print('Run kafka_producer.py in another terminal to send events:')
print('  docker compose exec spark-master python3 /app/scripts/kafka_producer.py')
console_query.awaitTermination(60)

26/05/06 22:10:08 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-5c9775ce-a376-4df7-aa32-86db300b8f0b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/06 22:10:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming query started — waiting 60 seconds for events...
Run kafka_producer.py in another terminal to send events:
  docker compose exec spark-master python3 /app/scripts/kafka_producer.py


26/05/06 22:10:10 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



26/05/06 22:10:31 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 21554 milliseconds
                                                                                

-------------------------------------------
Batch: 1
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



False

## 7. Write Stream to Parquet Sink (for dashboard + integration)

In [8]:
import os
os.makedirs('/data/streaming_output', exist_ok=True)
os.makedirs('/data/streaming_checkpoints/parquet', exist_ok=True)  # separate subfolder

parquet_query = (
    output_df
    .writeStream
    .outputMode('append')
    .format('parquet')
    .option('path', '/data/streaming_output/')
    .option('checkpointLocation', '/data/streaming_checkpoints/parquet/')  # own folder
    .trigger(processingTime='10 seconds')
    .queryName('parquet_sink')
    .start()
)

print('Parquet sink started → /data/streaming_output/')

Parquet sink started → /data/streaming_output/


26/05/06 22:11:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 8. Alert System

In [9]:
# Alert conditions:
#   1. avg_rating > 4.5  → item is receiving exceptionally high ratings
#   2. interaction_count > 50 → item is experiencing an activity spike in this window
alerts_df = output_df.filter(
    (col('avg_rating') > 4.5) | (col('interaction_count') > 50)
)

alert_query = (
    alerts_df
    .writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('checkpointLocation', '/data/streaming_checkpoints/alerts/')
    .trigger(processingTime='10 seconds')
    .queryName('alert_stream')
    .start()
)

print('Alert system active.')
print('  Triggers when: avg_rating > 4.5  OR  interaction_count > 50')
print('  Example output: ALERT — Item 12345 | avg_rating=4.8 | trending_score=240.0')

26/05/06 22:11:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/06 22:11:10 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Alert system active.
  Triggers when: avg_rating > 4.5  OR  interaction_count > 50
  Example output: ALERT — Item 12345 | avg_rating=4.8 | trending_score=240.0


## 9. Monitor Active Queries

In [10]:
print(f'Active streaming queries: {len(spark.streams.active)}')
for q in spark.streams.active:
    print(f'  - {q.name} | status: {q.status["message"]}')

Active streaming queries: 3
  - alert_stream | status: Initializing sources
  - console_output10 | status: Processing new data
  - parquet_sink | status: Getting offsets from KafkaV2[Subscribe[user_events]]


26/05/06 22:11:10 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 5
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 6
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



...

-------------------------------------------
Batch: 14
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 20
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 15
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 21
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



-------------------------------------------
Batch: 16
-------------------------------------------
+------------+----------+-------+----------+-----------------+--------------+
|window_start|window_end|item_id|avg_rating|interaction_count|trending_score|
+------------+----------+-------+----------+-----------------+--------------+
+------------+----------+-------+----------+-----------------+--------------+



In [6]:

# # Graceful stop — uncomment and run this block to stop all queries cleanly
# for q in spark.streams.active:
#     q.stop()
# spark.stop()
# print('All queries stopped.')

All queries stopped.
